# updatedatasettracker.ipynb

In [10]:
import pandas as pd
from sodapy import Socrata
import pandas as pd
import os
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', None)
import requests
import gspread
from oauth2client.service_account import ServiceAccountCredentials
bic_etl_home = os.getenv('bic_etl_home')

## Get Columns from Dataset Tracker

In [11]:
def getTrackerColumns():
    ''' Returns the columns in the Dataset Tracker PublishedData tab'''
    scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
                 "https://www.googleapis.com/auth/drive.file",
                      "https://www.googleapis.com/auth/drive"]

    #creds = ServiceAccountCredentials.from_json_keyfile_name('../../scripts/client_secret.json',
    #    scope)
    creds = ServiceAccountCredentials.from_json_keyfile_name(os.path.join(bic_etl_home, 'general', 'scripts','client_secret.json'),scope)

    client = gspread.authorize(creds)
    tracker = client.open('BIC Dataset Tracker').worksheet(
    'PublishedData')
    metadata_df = pd.DataFrame(tracker.get_all_records(head=2))
    return list(metadata_df.columns)

colsTracker = getTrackerColumns()

## Mapper Between Dataset Tracker and Metadata Form

In [32]:
#  Map tracker columns to metadata columns that have a different name
def mapper(colsTracker,colsMeta):
    # mapped = {"API Field Names, comma delimited":"Field Names, comma delimited",
    #  "Addt'l External Metadata Link":"Additional Metadata",
    #  "Geospatial Collection Method":"Collection Method",
    #  "Short Description":"Dataset Short Description"}

    
    # mapped = {"Field Names, comma delimited":"API Field Names, comma delimited",
    #  "Additional Metadata":"Addt'l External Metadata Link",
    #  "Collection Method":"Geospatial Collection Method",
    #  "Short Description":"Dataset Short Description"}
    mapped = {
     "Additional Metadata":"Addt'l External Metadata Link",
     "Collection Method":"Geospatial Collection Method"}                            

    
    mapping = {}
    for col in colsTracker:
        if col in colsMeta:
           mapping[col]=col
        elif col in mapped:
           mapping[col]=mapped[col]
    
    return mapping

## Read Metadata Forms and Create DataFrame

### Via Google API

In [33]:
scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
         "https://www.googleapis.com/auth/drive",
         "https://spreadsheets.google.com/feeds"
             ]
creds = ServiceAccountCredentials.from_json_keyfile_name('/home/joe/work/client_secret.json',scope)
client = gspread.authorize(creds)
gc = gspread.service_account("/home/joe/work/client_secret.json")
# for gg in gc.list_spreadsheet_files():
#      print("GGGGG ",gg)
    
try: 
    spread = client.open("Employment Counts Across Major Industry Sectors in Colorado")
    
    #Getting a list of worksheets inside a spreadsheet.
    sheets = spread.worksheets()
    nsheets=0
    tmp={}
    for sheet in sheets:
        if sheet.title == "metadata":
            nsheets+=1
      
            print(sheet.title)
            vals = sheet.get_all_values()
        #    tracker = client.open('Copy of Lobbyist Metadata').worksheet(sheet.name)
        #    df = pd.DataFrame(sheet.get_all_records(head=0))
        #    df = pd.DataFrame(sheet.get("A9:C50"),columns=["Name","Value","Description"])
            df = pd.DataFrame(vals[8:49],columns=["Name","Value","Description"])
            dfT = df.T    
            dfT.columns=dfT.loc['Name']
            dfT.drop(["Name","Description"],axis=0,inplace=True)
            dictMeta = dfT.to_dict()
            if nsheets == 1:
                mapping=mapper(colsTracker,list(dictMeta.keys()))
            fields={}
            if len(vals) > 53: 
                for row in vals[52:]:
                    fields[row[0].strip()] = row[1].strip()
            for col in colsTracker:
                print("COL ",col)
                if col not in tmp:
                    tmp[col]=[]
                if col in mapping:
                    val= dictMeta[mapping[col]]['Value']
                    if col == "Field Names, comma delimited" and len(fields.keys()) > 0: # add field names if found
                       val = ','.join(str(x) for x in list(fields.keys()))
                else:
                    val=""
                
                    
                tmp[col].append(val)
    
    #  Convert Dictionary to Dataframe            
    outDf = pd.DataFrame(tmp)
except Exception as err:
    print("ERROR ",err)


metadata
COL  Dataset Title
COL  Short Description
COL  Category
COL  Keywords
COL  Type
COL  License Type
COL  Data Provider
COL  Data Provided by
COL  Source Link
COL  State Steward
COL  Citation
COL  Agency Program Page
COL  Agency Data Series Page
COL  Business Contact and Phone
COL  Technical Contact and Phone
COL  Data Source
COL  Internal Source Name
COL  Unit of Analysis
COL  Geographic Extent and Division
COL  Collection Mode
COL  Collection Methodology
COL  Data Collection Instrument
COL  Field Names, comma delimited
COL  Oldest Record in Dataset
COL  Newest Record in Dataset
COL  Long Description
COL  Data Dictionary
COL  Additional Metadata
COL  Technical Documentation
COL  Data Quality Certification
COL  Applicable Information Quality Guideline Designation
COL  Stewardship Plan
COL  Collection Method
COL  Horizontal Accuracy
COL  Horizontal Coordinate System
COL  Update Schedule
COL  Update Method
COL  Source Update Schedule
COL  Update Schedule Last Audit
COL  Update Type

In [34]:
mapping 

{'Category': 'Category',
 'Keywords': 'Keywords',
 'License Type': 'License Type',
 'Data Provided by': 'Data Provided by',
 'Source Link': 'Source Link',
 'Citation': 'Citation',
 'Agency Program Page': 'Agency Program Page',
 'Agency Data Series Page': 'Agency Data Series Page',
 'Data Source': 'Data Source',
 'Unit of Analysis': 'Unit of Analysis',
 'Geographic Extent and Division': 'Geographic Extent and Division',
 'Collection Mode': 'Collection Mode',
 'Collection Methodology': 'Collection Methodology',
 'Data Collection Instrument': 'Data Collection Instrument',
 'Long Description': 'Long Description',
 'Data Dictionary': 'Data Dictionary',
 'Additional Metadata': "Addt'l External Metadata Link",
 'Technical Documentation': 'Technical Documentation',
 'Data Quality Certification': 'Data Quality Certification',
 'Applicable Information Quality Guideline Designation': 'Applicable Information Quality Guideline Designation',
 'Stewardship Plan': 'Stewardship Plan',
 'Collection Meth

In [36]:
outDf 

,Dataset Title,Short Description,Category,Keywords,Type,License Type,Data Provider,Data Provided by,Source Link,State Steward,Citation,Agency Program Page,Agency Data Series Page,Business Contact and Phone,Technical Contact and Phone,Data Source,Internal Source Name,Unit of Analysis,Geographic Extent and Division,Collection Mode,Collection Methodology,Data Collection Instrument,"Field Names, comma delimited",Oldest Record in Dataset,Newest Record in Dataset,Long Description,Data Dictionary,Additional Metadata,Technical Documentation,Data Quality Certification,Applicable Information Quality Guideline Designation,Stewardship Plan,Collection Method,Horizontal Accuracy,Horizontal Coordinate System,Update Schedule,Update Method,Source Update Schedule,Update Schedule Last Audit,Update Type,Total Records at Initial Publish,Row Class RDF,Subject Column RDF,Single Row,Total Fields at Initial Publish,Expected approximate increase in record count at update,Date Published to CIM,GoCode FY Published to CIM,Socrata Link,MapViz,Web Display Coordinate System,Coordinate System Disclaimer,Related Datasets,Quarter of Gov FY Published,cimAllData Updates,Complexity,Business Contact Information - Name,Business Contact Information - Position,Business Contact Information - Phone Number,Business Contact Information - Email,Technical Contact Information - Name,Technical Contact Information - Position,Technical Contact Information - Phone Number,Technical Contact Information - Email,Name of Person Providing Data,Email of Person Providing Data
0,,,Labor & Employment,"Colorado, employment, industry, non-agricultur...",,Public Domain,,Bureau of Labour Statistics,https://data.bls.gov/PDQWeb/sm,,Bureau of Labour Statistics,https://www.bls.gov/sae/home.htm,https://download.bls.gov/pub/time.series/sm/,,,,,,,,Survey,,,,,BLS collects data each month from a sample of ...,https://download.bls.gov/pub/time.series/sm/sm...,http://www.bls.gov/opub/hom/pdf/homch2.pdf,,,,,,,,Automated-monthly,Automated by Staging Server,monthly,,Automated,,,,,5,,,,,,,,,,,,,,,,,,,,,


In [4]:
from googleapiclient.discovery import build
from oauth2client.service_account import ServiceAccountCredentials

# Define the scope
scope = [
    "https://www.googleapis.com/auth/spreadsheets.readonly",
    "https://www.googleapis.com/auth/drive.readonly"
]

# Add your service account credentials
creds = ServiceAccountCredentials.from_json_keyfile_name("/home/joe/work/client_secret.json", scope)

# Build the Google Drive service
drive_service = build('drive', 'v3', credentials=creds)

# List files in Google Drive
results = drive_service.files().list(pageSize=10, fields="nextPageToken, files(id, name)").execute()
items = results.get('files', [])

if not items:
    print('No files found.')
else:
    print('Files:')
    for item in items:
        print(f'{item["name"]} ({item["id"]})')

# # Access the Google Sheets service
# sheets_service = build('sheets', 'v4', credentials=creds)

# # Open the spreadsheet by ID (replace 'spreadsheet_id' with the actual ID)
# spreadsheet_id = 'your_spreadsheet_id'
# spreadsheet = sheets_service.spreadsheets().get(spreadsheetId=spreadsheet_id).execute()

# # Print the spreadsheet title
# print('Spreadsheet title:', spreadsheet['properties']['title'])

Files:
Area Employment, Hours, and Earnings Metadata.xlsx (159Ml9n7DH17IBqS0tvEfQp5zuOAsmA4G)
Copy of Monthly Consumer Price Index, Denver-Aurora-Lakewood Metadata Form.xlsx (195LVxM2oo9whtot5Bsj9aAadclPMxGNC)
Copy of Lobbyist Metadata (1rGj3iTOFa4zZanCxqPi6Nklb3XOo0OyhTDPzagEm7vE)
Copy of Area Employment, Hours, and Earnings Metadata.xlsx (119tiI-1WPNX-8l62-WtZWLVxoiOh9UYt)
automatedStatus.png (1vgPTEIOckaAwGA_fao9IQYLB0_29Tw3V)
ETL Errors Accounted For (1TATw3NXZwb7WseBVRtlIYdhVkLJRfS7jBT3jU1v0340)
Changes/Fixes Requested by BIC (1ORf8R6_inddQpltzb6EEW3agaXnIR0z2SBNUJZH2WYA)
Copy of bic_metadata_updater (16ji8A-62nOiPwjBLVkOKAWkGEbzhi3D3byl1eunOu30)


### Via Excel Spreadshet

In [30]:
df=pd.read_excel("Monthly Consumer Price Index, Denver-Aurora-Lakewood Metadata Form.xlsx")
df=df[5:]
df.columns=["Name","Value","Description"]

In [31]:
df.head()

,Name,Value,Description
5,Dataset Title,"Monthly Consumer Price Index, Denver-Aurora-La...",{[3 word description] + [data type] + [geograp...
6,Dataset Short Description,This dataset provides the Consumer Price Inde...,{Whats in it; who is the provider; what is the...
7,Category,Business,"[Agriculture, Business, Demographics, Economic..."
8,Keywords,"CPI-U, inflation, consumer price index, Denver...",{The best keywords are succinct and unambiguou...
9,Row Label,A single row represents CPI-U percent change f...,{Describe what a single row in the data repres...


In [32]:
dfT = df.T    
dfT.columns=dfT.loc['Name']
dfT.drop(["Name","Description"],axis=0,inplace=True)
dictMeta = dfT.to_dict()

print(dictMeta)

{'Dataset Title': {'Value': 'Monthly Consumer Price Index, Denver-Aurora-Lakewood '}, 'Dataset Short Description': {'Value': 'This dataset provides the  Consumer Price Index (CPI) for the Denver-Aurora-Lakewood area, with some CPI\'s reporting monthly and some reporting only on the odd numbered months.  CPI is a statistical measure\nof change, over time, of the prices of goods and services in major \nexpenditure groups--such as food, housing, apparel, transportation, and \nmedical care--typically purchased by urban consumers. Essentially, it \ncompares the cost of a sample "market basket" of goods and services in a specific month relative to the cost of the same "market basket" in an \nearlier reference period. This reference period is designated as the base \nperiod.'}, 'Category': {'Value': 'Business'}, 'Keywords': {'Value': 'CPI-U, inflation, consumer price index, Denver-Aurora-Lakewood, price change, cost of living, economic indicators, monthly data, All Items, All Items Less Food 

In [33]:
print(dictMeta)

{'Dataset Title': {'Value': 'Monthly Consumer Price Index, Denver-Aurora-Lakewood '}, 'Dataset Short Description': {'Value': 'This dataset provides the  Consumer Price Index (CPI) for the Denver-Aurora-Lakewood area, with some CPI\'s reporting monthly and some reporting only on the odd numbered months.  CPI is a statistical measure\nof change, over time, of the prices of goods and services in major \nexpenditure groups--such as food, housing, apparel, transportation, and \nmedical care--typically purchased by urban consumers. Essentially, it \ncompares the cost of a sample "market basket" of goods and services in a specific month relative to the cost of the same "market basket" in an \nearlier reference period. This reference period is designated as the base \nperiod.'}, 'Category': {'Value': 'Business'}, 'Keywords': {'Value': 'CPI-U, inflation, consumer price index, Denver-Aurora-Lakewood, price change, cost of living, economic indicators, monthly data, All Items, All Items Less Food 

In [34]:
mapping=mapper(colsTracker,list(dictMeta.keys()))
fields={}
tmp={}
for col in colsTracker:
    if col not in tmp:
        tmp[col]=[]
    if col in mapping:
        if col != "Field Names, comma delimited": # add field names if found
          val= dictMeta[mapping[col]]['Value']
 #         print(col," : ",val)
        else:
          val=""
          
    else:
        val=""  
    tmp[col].append(val)

#  Convert Dictionary to Dataframe            
outDf = pd.DataFrame(tmp)
outDf=outDf.fillna('')

In [35]:
colsTracker

['Dataset Title',
 'Short Description',
 'Category',
 'Keywords',
 'Type',
 'License Type',
 'Data Provider',
 'Data Provided by',
 'Source Link',
 'State Steward',
 'Citation',
 'Agency Program Page',
 'Agency Data Series Page',
 'Business Contact and Phone',
 'Technical Contact and Phone',
 'Data Source',
 'Unit of Analysis',
 'Granularity Coverage',
 'Geographic Extent and Division',
 'Collection Mode',
 'Collection Methodology',
 'Data Collection Instrument',
 'Date of Initial Dataset Creation',
 'Field Names, comma delimited',
 'Oldest Record in Dataset',
 'Newest Record in Dataset',
 'Long Description',
 'Data Dictionary',
 'Additional Metadata',
 'Technical Documentation',
 'Data Quality Certification',
 'Applicable Information Quality Guideline Designation',
 'Stewardship Plan',
 'Collection Method',
 'Horizontal Accuracy',
 'Horizontal Coordinate System',
 'Update Schedule',
 'Update Method',
 'Source Update Schedule',
 'Update Schedule Last Audit',
 'Update Type',
 'Total Rec

In [36]:
outCols=list(outDf.columns)

for nn,col in enumerate(outCols):
    print(col)
    print(outCols[nn])
    print()
    

Dataset Title
Dataset Title

Short Description
Short Description

Category
Category

Keywords
Keywords

Type
Type

License Type
License Type

Data Provider
Data Provider

Data Provided by
Data Provided by

Source Link
Source Link

State Steward
State Steward

Citation
Citation

Agency Program Page
Agency Program Page

Agency Data Series Page
Agency Data Series Page

Business Contact and Phone
Business Contact and Phone

Technical Contact and Phone
Technical Contact and Phone

Data Source
Data Source

Unit of Analysis
Unit of Analysis

Granularity Coverage
Granularity Coverage

Geographic Extent and Division
Geographic Extent and Division

Collection Mode
Collection Mode

Collection Methodology
Collection Methodology

Data Collection Instrument
Data Collection Instrument

Date of Initial Dataset Creation
Date of Initial Dataset Creation

Field Names, comma delimited
Field Names, comma delimited

Oldest Record in Dataset
Oldest Record in Dataset

Newest Record in Dataset
Newest Record in

## Output DataFrame as Excel File

In [37]:
outDf.to_excel("bls.cu.montly.xlsx",index=False)

## Field Names and Descriptions

In [ ]:
scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
         "https://www.googleapis.com/auth/drive.file",
              "https://www.googleapis.com/auth/drive"]
creds = ServiceAccountCredentials.from_json_keyfile_name('/home/joe/work/client_secret.json',scope)
client = gspread.authorize(creds)
spread = client.open("Copy of Lobbyist Metadata")

#Getting a list of worksheets inside a spreadsheet.
sheets = spread.worksheets()
nsheets=0
tmp={}
for sheet in sheets:
    if sheet.title != "Example_Directory of Lobbyists":
        nsheets+=1
  
        print(sheet.title)
        vals = sheet.get_all_values()
        fields={}
        if len(vals) > 53: 
            for row in vals[52:]:
                fields[row[0].strip()] = row[1].strip()
        file = sheet.title.replace(" ","")
        print(file)
        fout=open(f"{file}-fields.tsv","w")
        for key,val in fields.items():
            key=key.replace("\t","")
            val=val.replace("\t","")
            
            fout.write(f"{key}\t{val}\n")
        fout.close()
       



colsTracker -> Columns in Dataset Tracker Sheet<br>
colsMeta    -> Columns in Metadata Form we send out or use